Cost per million tokens formula:
GPU hourly cost / (tokens_per_s × 3600 / 1,000,000)

Prediction:
If I need roughly double the safe throughput, I expect adding a second replica
at the same knee concurrency to be the better option if pushing one GPU further
causes p95 latency to violate the SLO.

In [23]:
import json

levels = json.load(open("/content/bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"p95={L['latency_p95_s']:.3f}s  "
        f"errors={L['errors']}"
    )

c= 1  tok/s=   57.7  p95=1.313s  errors=0
c= 2  tok/s=  105.0  p95=1.287s  errors=0
c= 4  tok/s=  206.9  p95=1.287s  errors=0
c= 8  tok/s=  330.6  p95=1.314s  errors=0
c=16  tok/s=  446.7  p95=1.533s  errors=0


In [24]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(
        L["tokens_per_s"],
        GPU_HOURLY_USD
    )

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"p95={L['latency_p95_s']:.2f}s  "
        f"$/M tok=${L['cost_per_million_tokens_usd']}"
    )

c= 1  tok/s=   57.7  p95=1.31s  $/M tok=$1.6853
c= 2  tok/s=  105.0  p95=1.29s  $/M tok=$0.9257
c= 4  tok/s=  206.9  p95=1.29s  $/M tok=$0.47
c= 8  tok/s=  330.6  p95=1.31s  $/M tok=$0.2941
c=16  tok/s=  446.7  p95=1.53s  $/M tok=$0.2176


In [25]:
TARGET_P95_S = 3.0

under_target = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under_target,
    key=lambda L: L["concurrency"]
) if under_target else None

print("knee:", knee)

past_knee = [
    L for L in levels
    if knee and L["concurrency"] > knee["concurrency"]
]

if past_knee:
    cheapest_past_knee = min(
        past_knee,
        key=lambda L: L["cost_per_million_tokens_usd"]
    )
    print(
        "cheapest $/M token level past the knee (SLO-violating):",
        cheapest_past_knee
    )
else:
    print("No tested level is past the knee; knee is sweep-bounded.")

knee: {'concurrency': 16, 'tokens_per_s': 446.7431738447158, 'ttft_p50_s': 0.17405440399988947, 'ttft_p95_s': 0.17774163115013833, 'latency_p95_s': 1.5334834780003574, 'errors': 0, 'cost_per_million_tokens_usd': 0.2176}
No tested level is past the knee; knee is sweep-bounded.


In [33]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s - 1e-9)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],
    }

targets = [
    round(knee["tokens_per_s"] * m, 6)
    for m in (1.0, 1.5, 2.0, 3.0)
]

scale_plan = [
    scale_out_cost(t, knee, GPU_HOURLY_USD)
    for t in targets
]

for row in scale_plan:
    print(row)

{'required_tokens_per_s': 446.743174, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 1.5334834780003574}
{'required_tokens_per_s': 670.114761, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.5334834780003574}
{'required_tokens_per_s': 893.486348, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 1.5334834780003574}
{'required_tokens_per_s': 1340.229522, 'replicas_needed': 4, 'total_hourly_cost_usd': 1.4, 'effective_p95_s': 1.5334834780003574}


In [34]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}

with open("/content/cost_report.json", "w") as f:
    json.dump(report, f, indent=2)

In [35]:
!python /content/verify.py

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
